In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import glob
from datetime import datetime, timedelta
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)


def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    return ds_djf


def calculate_tercile_flags(data, var):
    # calculate terciles to cluster indices into 3 categories: -1 (negative/cold), 0 (netural), 1 (positive/warm state)
    df = data[var].to_dataframe()
    df = df.reset_index()
    df = df.set_index('time')
    tertiles = df[var].quantile([1/3, 2/3]).to_list()
    tertiles = [-float('inf')] + tertiles + [float('inf')]

    df['tercile'] = pd.cut(df[var], bins=tertiles, labels=[-1,0,1])
    return df['tercile'], tertiles

def calculate_index(data, var, time_coder=time_coder, sel_time=None):
    # calculate index and tercile flags for a given variable and path (ENSO: var='tos', NAO: var='zg')
    if sel_time is not None:
        data = data.isel(time=sel_time)
    index = (data - data.mean('time'))/ np.sqrt(data.var('time'))   
    index['tercile'], tertiles = calculate_tercile_flags(index, var)
    return index, tertiles


In [2]:
# prepare dataframe
df_all = []
outfilename = "/home/kperny/VolMIP_stratosphere/tab/nao_flags.csv"

In [3]:
###########
# CanESM5 #
###########
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/CanESM5/NAO*"

path_nao = "/sto4/data/Intermediate/VolMIP/piControl/CanESM5/NAO_Amon_piControl.nc"
data = xr.open_dataset(path_nao, decode_times=time_coder).squeeze()
data_djf = calculate_djf_mean(data)
nao, tertiles = calculate_index(data_djf, 'zg')

realization_can = []
nao_pi = []
nao_flag_pi = []
nao_volc = []
nao_flag_volc = []
year_can = []
for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization_can.append(file.split('/')[-1].split('_')[4])
    data_can = xr.open_dataset(file)
    year_post_eruption = str(int(data_can.attrs['YMDH_branch_time_in_parent'].split(':')[0]) + 1)
    year_can.append(str(year_post_eruption) + '-02-01')

    nao_pi.append(nao.sel(time=year_post_eruption).zg.values[0])
    data = (calculate_djf_mean(data_can).sel(time='1992') - data_djf.mean('time')) /  np.sqrt(data_djf.var('time'))
    nao_volc.append(float(data.zg.squeeze().values))
    nao_flag_pi.append(nao.sel(time=year_post_eruption).tercile.values[0])
df = pd.DataFrame({'realization': realization_can, 'year': year_can, 'nao_pi': nao_pi, 'flag_pi': nao_flag_pi, 'nao_volc': nao_volc})
df['flag_volc'] = pd.cut(df['nao_volc'], bins=tertiles, labels=[-1,0,1])
df['model'] = 'CanESM5'
df_all.append(df)

In [4]:
###############
# GISS-E2-1-G #
###############
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/GISS-E2-1-G/NAO*"
path_nao = "/sto4/data/Intermediate/VolMIP/piControl/GISS-E2-1-G/NAO_Amon_piControl.nc"
data = xr.open_dataset(path_nao, decode_times=time_coder).squeeze()
data_djf = calculate_djf_mean(data)
nao, tertiles = calculate_index(data_djf, 'zg')

realization_giss = []
nao_pi =[]
nao_volc =[]
nao_flag_pi = []
year_giss  = []
for file in sorted(glob.glob(path)):
    # file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization_giss.append(file.split('/')[-1].split('_')[4])
    data_giss = xr.open_dataset(file, decode_times=time_coder)    
    year_post_eruption = str(int(str(data_giss.time[0].values).split('-')[0]) + 1)
    year_giss.append(str(year_post_eruption) + '-02-01')
    
    nao_pi.append(nao.sel(time=year_post_eruption).zg.values[0])
    nao_flag_pi.append(nao.sel(time=year_post_eruption).tercile.values[0])

    data_giss['time'] = pd.date_range(start='1991-01-01', end='1993-12-01', freq='MS')
    data = (calculate_djf_mean(data_giss).sel(time='1992') - data_djf.mean('time')) /  np.sqrt(data_djf.var('time'))
    nao_volc.append(float(data.zg.squeeze().values))
df = pd.DataFrame({'realization': realization_giss, 'year': year_giss, 'nao_pi': nao_pi, 'flag_pi': nao_flag_pi, 'nao_volc': nao_volc})
df['flag_volc'] = pd.cut(df['nao_volc'], bins=tertiles, labels=[-1,0,1])
df['model'] = 'GISS-E2-1-G'
df_all.append(df)


In [5]:
################
# IPSL-CM6A-LR #
################
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/IPSL-CM6A-LR/NAO*"
path_nao = "/sto4/data/Intermediate/VolMIP/piControl/IPSL-CM6A-LR/NAO_Amon_piControl.nc"
data = xr.open_dataset(path_nao, decode_times=time_coder).squeeze()
data_djf = calculate_djf_mean(data)
nao, tertiles = calculate_index(data_djf, 'zg')

realization_ipsl = []
nao_pi =[]
nao_volc =[]
nao_flag_pi = []
year_ipsl = []
for file in sorted(glob.glob(path)):
    # file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization_ipsl.append(file.split('/')[-1].split('_')[3])
    data_ipsl = xr.open_dataset(file, decode_times=time_coder)    
    start = datetime.strptime(data_ipsl.attrs['parent_time_units'].split(' ')[2], '%Y-%m-%d') 
    year_post_eruption = str((start + timedelta(days=data_ipsl.attrs['branch_time_in_parent'])).year + 1)
    year_ipsl.append(str(year_post_eruption) + '-02-01')
    
    nao_pi.append(nao.sel(time=year_post_eruption).zg.values[0])
    nao_flag_pi.append(nao.sel(time=year_post_eruption).tercile.values[0])

   # data_ipsl['time'] = pd.date_range(start='1991-01-01', end='1993-12-01', freq='MS')
    data = (calculate_djf_mean(data_ipsl).sel(time='1992') - data_djf.mean('time')) /  np.sqrt(data_djf.var('time'))
    nao_volc.append(float(data.zg.squeeze().values))
df = pd.DataFrame({'realization': realization_ipsl, 'year': year_ipsl, 'nao_pi': nao_pi, 'flag_pi': nao_flag_pi, 'nao_volc': nao_volc})
df['flag_volc'] = pd.cut(df['nao_volc'], bins=tertiles, labels=[-1,0,1])
df['model'] = 'IPSL-CM6A-LR'
df_all.append(df)

In [6]:
##############
# MIROC-ES2L #
##############
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/MIROC-ES2L/NAO*"

path_nao = "/sto4/data/Intermediate/VolMIP/piControl/MIROC-ES2L/NAO_Amon_piControl.nc"

data = xr.open_dataset(path_nao, decode_times=time_coder).squeeze()
data_djf = calculate_djf_mean(data)
nao, tertiles = calculate_index(data_djf, 'zg')

realization_miroc = []
nao_pi =[]
nao_volc =[]
nao_flag_pi = []
year_miroc = []
for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization_miroc.append(file.split('/')[-1].split('_')[4])
    data_miroc = xr.open_dataset(file, decode_times=time_coder)
    start = datetime.strptime(data_miroc.attrs['parent_time_units'].split(' ')[2], '%Y-%m-%d') 
    year_post_eruption = str((start + timedelta(days=data_miroc.attrs['branch_time_in_parent'])).year + 1)
    year_miroc.append(str(year_post_eruption) + '-02-01')
    
    nao_pi.append(nao.sel(time=year_post_eruption).zg.values[0])
    nao_flag_pi.append(nao.sel(time=year_post_eruption).tercile.values[0])

    data_miroc['time'] = pd.date_range(start='1991-06-01', end='1994-12-01', freq='MS')
    data = (calculate_djf_mean(data_miroc).sel(time='1992') - data_djf.mean('time')) /  np.sqrt(data_djf.var('time'))
    nao_volc.append(float(data.zg.squeeze().values))

df = pd.DataFrame({'realization': realization_miroc, 'year': year_miroc, 'nao_pi': nao_pi, 'flag_pi': nao_flag_pi, 'nao_volc': nao_volc})
df['flag_volc'] = pd.cut(df['nao_volc'], bins=tertiles, labels=[-1,0,1])
df['model'] = 'MIROC-ES2L'
df_all.append(df)

In [7]:
#################
# MPI-ESM1-2-LR #
#################
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/MPI-ESM1-2-LR/NAO*"
path_nao = "/sto4/data/Intermediate/VolMIP/piControl/MPI-ESM1-2-LR/NAO_Amon_piControl.nc"

data = xr.open_dataset(path_nao, decode_times=time_coder).squeeze()
data_djf = calculate_djf_mean(data)
nao, tertiles = calculate_index(data_djf, 'zg')

realization_mpi = []
nao_pi =[]
nao_volc =[]
nao_flag_pi = []
year_mpi = []
for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization_mpi.append(file.split('/')[-1].split('_')[4])
    data_mpi = xr.open_dataset(file, decode_times=time_coder)
    start = datetime.strptime(data_mpi.attrs['parent_time_units'].split(' ')[2], '%Y-%m-%d') 
    year_post_eruption = str((start + timedelta(days=data_mpi.attrs['branch_time_in_parent'])).year + 1)
    year_mpi.append(str(year_post_eruption) + '-02-01')
    
    nao_pi.append(nao.sel(time=year_post_eruption).zg.values[0])
    nao_flag_pi.append(nao.sel(time=year_post_eruption).tercile.values[0])

    data_mpi['time'] = data_mpi.indexes['time'].to_datetimeindex(time_unit='ns', unsafe=True).map(lambda t: t.replace(day=1, hour=0))
    data = (calculate_djf_mean(data_mpi).sel(time='1992') - data_djf.mean('time')) /  np.sqrt(data_djf.var('time'))
    nao_volc.append(float(data.zg.squeeze().values))

df = pd.DataFrame({'realization': realization_mpi, 'year': year_mpi, 'nao_pi': nao_pi, 'flag_pi': nao_flag_pi, 'nao_volc': nao_volc})
df['flag_volc'] = pd.cut(df['nao_volc'], bins=tertiles, labels=[-1,0,1])
df['model'] = 'MPI-ESM1-2-LR'
df_all.append(df)


In [8]:
###############
# UKESM1-0-LL #
###############
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/UKESM1-0-LL/NAO*"
path_nao = "/sto4/data/Intermediate/VolMIP/piControl/UKESM1-0-LL/NAO_Amon_piControl.nc"

data = xr.open_dataset(path_nao, decode_times=time_coder).squeeze()
data_djf = calculate_djf_mean(data)
nao, tertiles = calculate_index(data_djf, 'zg')

# hard-coded years from Zanchettin et al 2022, Table S1, as the branch time in parent and child attributes are not correct for UKESM1-0-LL
years = [3174, 3064, 3067, 3128, 3116, 3171, 3138, 3153, 3052, 3135, 3141, 3193, 3058, 3055, 3061, 3085, 3094, 3049, 3177, 3165, 3122, 3073, 3103, 3076, 3180]

realization_ukesm = []
nao_pi =[]
nao_volc =[]
nao_flag_pi = []
year_ukesm = []
#for file in sorted(glob.glob(path)):
for i, year in enumerate(years):
    file = glob.glob(path + 'r' + str(i + 1) + 'i1p1f2*')[0]
    realization_ukesm.append(file.split('/')[-1].split('_')[4])
    data_ukesm = xr.open_dataset(file, decode_times=time_coder)
    
    year_post_eruption = str(year + 1)
    year_ukesm.append(str(year_post_eruption) + '-02-01')
    
    nao_pi.append(nao.sel(time=year_post_eruption).zg.values[0])
    nao_flag_pi.append(nao.sel(time=year_post_eruption).tercile.values[0])

    data_ukesm['time'] = data_ukesm.indexes['time'].to_datetimeindex(time_unit='ns', unsafe=True).map(lambda t: t.replace(day=1, hour=0))
    data = (calculate_djf_mean(data_ukesm).sel(time='1992') - data_djf.mean('time')) /  np.sqrt(data_djf.var('time'))
    nao_volc.append(float(data.zg.squeeze().values))

df = pd.DataFrame({'realization': realization_ukesm, 'year': year_ukesm, 'nao_pi': nao_pi, 'flag_pi': nao_flag_pi, 'nao_volc': nao_volc})
df['flag_volc'] = pd.cut(df['nao_volc'], bins=tertiles, labels=[-1,0,1])
df['model'] = 'UKESM1-0-LL'
df_all.append(df)

In [10]:
# concatenate all dataframes and save as csv
pd.concat(df_all).to_csv(outfilename, index=False)